In [1]:
!python3 -V
import pDESy
from IPython.display import Markdown
from pDESy.model.base_project import BaseProject
from pDESy.model.base_project import datetime
from pDESy.model.base_product import BaseProduct
from pDESy.model.base_component import BaseComponent
from pDESy.model.base_workflow import BaseWorkflow
from pDESy.model.base_task import BaseTask
from pDESy.model.base_task import BaseTaskDependency
from pDESy.model.base_team import BaseTeam
from pDESy.model.base_worker import BaseWorker
from pDESy.model.base_facility import BaseFacility
from pDESy.model.base_workplace import BaseWorkplace
from pDESy.model.base_priority_rule import TaskPriorityRuleMode,ResourcePriorityRuleMode

Python 3.13.5


In [2]:
pDESy.__version__
pDESy.__file__

'/Users/keisukehirukawa/dev/pDESy_v0.7.3/pDESy/__init__.py'

製品定義

In [3]:
project = BaseProject("sample_workflow")
project = BaseProject(init_datetime = datetime.datetime(2025, 1, 1, 0, 0, 0), unit_timedelta=datetime.timedelta(minutes=60))

# product = project.create_product("product")
for i in range(1):
    command = f"product = project.create_product('product')"
    exec(command)

A = product.create_component("A")
B = product.create_component("B")

time = project.create_workflow("time")

ワークフロー定義・同一ワークフロー間依存関係

In [4]:
workflowA = project.create_workflow("workflowA")

task_A1 = workflowA.create_task("A1", need_facility=True, default_work_amount=6.0)
task_A1_2 = workflowA.create_task("A1_2", need_facility=True, default_work_amount=6.0)
task_A2 = workflowA.create_task("A2", need_facility=True, default_work_amount=6.0)
task_A2_2 = workflowA.create_task("A2_2", need_facility=True, default_work_amount=6.0)
task_A3 = workflowA.create_task("A3", need_facility=True, default_work_amount=6.0)

A.update_targeted_task_set({task_A1, task_A1_2, task_A2, task_A2_2, task_A3})

In [5]:

day1_5am = time.create_task(name="Waiting Task", default_work_amount=5, auto_task=True)
A1_inst_start = time.create_task(name="A1 Inst Start", default_work_amount=0, auto_task=True)
# A1_inst_end = time.create_task(name="A1 Inst End", default_work_amount=0, auto_task=True)

A1_inst_start.add_input_task(day1_5am)
# A1_inst_end.add_input_task(A1_inst_start)

task_A1_2.add_input_task(task_A1, BaseTaskDependency.SS)
task_A1.add_input_task(A1_inst_start, BaseTaskDependency.SS)
# A1_inst_end.update_input_task_set({task_A1, task_A1_2}, BaseTaskDependency.FF)

A2_inst_start = time.create_task(name="A2 Inst Start", default_work_amount=0, auto_task=True)
# A2_inst_end = time.create_task(name="A2 Inst End", default_work_amount=0, auto_task=True)
A2_inst_start.update_input_task_set({task_A1, task_A1_2}, BaseTaskDependency.FS)
# A2_inst_end.add_input_task(A2_inst_start)

# A2_inst_start.add_input_task(A1_inst_end, BaseTaskDependency.FS)

task_A2.add_input_task(A2_inst_start, BaseTaskDependency.SS)
task_A2_2.add_input_task(task_A2, BaseTaskDependency.SS)
# A2_inst_end.update_input_task_set({task_A2, task_A2_2}, BaseTaskDependency.FF)

task_A3.add_input_task(task_A2)

In [6]:
# workflowB = project.create_workflow("workflowB")

# task_B1 = workflowB.create_task("B1", need_facility=True, default_work_amount=24.0)
# task_B2 = workflowB.create_task("B2", need_facility=True, default_work_amount=72.0)
# task_B3 = workflowB.create_task("B3", need_facility=True, default_work_amount=24.0)

# B.update_targeted_task_set({task_B1, task_B2, task_B3})

# task_B2.add_input_task(task_B1)
# task_B3.add_input_task(task_B2)

In [7]:
# workflowC = project.create_workflow("workflowC")

# task_C1 = workflowC.create_task("C1", need_facility=True, default_work_amount=3.0)
# task_C2 = workflowC.create_task("C2", need_facility=True, default_work_amount=3.0)
# task_C3 = workflowC.create_task("C3", need_facility=True, default_work_amount=3.0)

# B.update_targeted_task_set({task_C1, task_C2, task_C3})

# task_C2.add_input_task(task_C1)
# task_C3.add_input_task(task_C2)

異なるワークフロー間依存関係

In [8]:
# task_B1.add_input_task(task_A2)

# task_C2.add_input_task(task_B1, BaseTaskDependency.SF)

設備・人員

In [9]:
# wrokplace model
placeA = project.create_workplace("placeA", max_space_size=10.0)
# placeB = project.create_workplace("placeB", max_space_size=10.0)

facilityA = placeA.create_facility("facilityA", cost_per_time=1)
facilityA.workamount_skill_mean_map = {task_A1.name:1.0, task_A2.name:1.0} 
facilityA.workamount_skill_mean_map.update({task_A3.name:1.0})
facilityA_2 = placeA.create_facility("facilityA_2", cost_per_time=1)
facilityA_2.workamount_skill_mean_map = {task_A1_2.name:1.0, task_A2_2.name:1.0} 
facilityA_2.workamount_skill_mean_map.update({task_A3.name:1.0})

# facilityB = placeB.create_facility("facilityB", cost_per_time=1)
# facilityB.workamount_skill_mean_map = {task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0} 

# facilityC = placeB.create_facility("facilityC", cost_per_time=1)
# facilityC.workamount_skill_mean_map = {task_C1.name:1.0,task_C2.name:1.0, task_C3.name:1.0}

#team model
team = project.create_team("team1")

wA = team.create_worker("workerA", cost_per_time=10.0)
wB = team.create_worker("workerB", cost_per_time=10.0)
# wC = team.create_worker("workerC", cost_per_time=10.0)

wA.workamount_skill_mean_map = {
    task_A1.name:1.0, task_A1_2.name:1.0, task_A2.name:1.0, task_A2_2.name:1.0, task_A3.name:1.0,
    # task_B1.name:1.0,task_B2.name:1.0,
    } 
wB.workamount_skill_mean_map = {
    task_A1.name:1.0, task_A1_2.name:1.0, task_A2.name:1.0,
    # task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0
    }
# wC.workamount_skill_mean_map = {
#     task_C1.name:1.0,task_C2.name:1.0, task_C3.name:1.0
#     }

wA.facility_skill_map = {facilityA.name:1.0, facilityA_2.name:1.0, 
                        #  facilityB.name:1.0
                        }
wB.facility_skill_map = wA.facility_skill_map.copy()
# wC.facility_skill_map = {facilityC.name:1.0}

team.update_targeted_task_set({task_A1,task_A1_2, task_A2, task_A2_2, task_A3, 
                            #    task_B1,task_B2,task_B3
                            })

placeA.update_targeted_task_set({task_A1, task_A1_2, task_A2, task_A2_2, task_A3})
# placeB.update_targeted_task_set({task_B1,task_B2,task_B3})

In [10]:
print(facilityA.workamount_skill_mean_map)

{'A1': 1.0, 'A2': 1.0, 'A3': 1.0}


In [11]:
print(wA.workamount_skill_mean_map)
print(wA.facility_skill_map)

{'A1': 1.0, 'A1_2': 1.0, 'A2': 1.0, 'A2_2': 1.0, 'A3': 1.0}
{'facilityA': 1.0, 'facilityA_2': 1.0}


In [12]:
project.simulate(max_time=600, progress_bar=True)

Completed:   4%|▍         | 23/600 [00:00<00:00, 13633.27time/s] 


In [13]:
import plotly.figure_factory as ff
from plotly.figure_factory import create_gantt

workflow_list = [time, workflowA]
task_id_list = [
    day1_5am.ID,
    A1_inst_start.ID,A2_inst_start.ID,
    task_A1.ID, task_A1_2.ID, task_A2.ID, task_A2_2.ID, task_A3.ID,
    # task_B1.ID, task_B2.ID, task_B3.ID,
    # task_C1.ID, task_C2.ID, task_C3.ID
]

combined_df = []
for wf in workflow_list:
    combined_df.extend(
        wf.create_data_for_gantt_plotly(
            init_datetime=project.init_datetime,
            unit_timedelta=project.unit_timedelta,
            target_id_order_list=list(task_id_list),
            view_ready=False,           # READY も表示したいなら True
            finish_margin=1.0
        )
    )

colors = {"WORKING": "rgb(0, 149, 255)", "READY": "rgb(107,127,135)"}

fig = create_gantt(
    combined_df,
    title="Gantt",
    colors=colors,
    index_col="State",
    showgrid_x=True,
    showgrid_y=True,
    group_tasks=True,
    show_colorbar=True,
)

fig.show()

In [15]:
workflowA.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [16]:
workflowB.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

NameError: name 'workflowB' is not defined

In [ ]:
from plotly.figure_factory import create_gantt

task_id_list = [
    task_A1.ID, task_A2.ID, task_A3.ID,
    task_B1.ID, task_B2.ID, task_B3.ID
]

all_workflows = [
    workflowA,
    workflowB
]

combined_df = []
for wf in all_workflows:
    combined_df.extend(
        wf.create_data_for_gantt_plotly(
            init_datetime=project.init_datetime,
            unit_timedelta=project.unit_timedelta,
            target_id_order_list=list(task_id_list),
            print_workflow_name=True,
            view_ready=False,           # READY も表示したいなら True
            finish_margin=1.0
        )
    )

colors = {"WORKING": "rgb(146, 237, 5)", "READY": "rgb(107,127,135)"}

fig = create_gantt(
    combined_df,
    title="All Workflows Gantt",
    colors=colors,
    index_col="State",
    showgrid_x=True,
    showgrid_y=True,
    group_tasks=True,
    show_colorbar=True,
)

fig.show()

In [ ]:
team.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [ ]:
diagram = "flowchart TB\n" + "\n".join(project.get_mermaid_diagram())
display(Markdown(f"```mermaid\n{diagram}\n```"))

```mermaid
flowchart TB
subgraph 86fee451-d400-4f10-bff2-a81a34afbf60[product]
direction LR
dc27aec0-93e7-485e-8297-476ab7288297@{shape: odd, label: 'A'}
de4fa2ec-b69f-4cb0-a1e7-685a13c1afe3@{shape: odd, label: 'B'}
end
subgraph 2407eaad-d13a-412a-9e56-b58d00a3366a[workflowA]
direction LR
4a8cdef8-984e-49eb-9e55-ecb7fc0f6ebf@{shape: rect, label: 'A2_2<br>0.0'}
e7c16258-5465-4a41-a7a7-68cba0a632f3@{shape: rect, label: 'A2<br>0.0'}
93e3fcf1-3a1e-490e-8d29-57423e576837@{shape: rect, label: 'A1<br>0.0'}
6c658872-dc91-4e9a-b5fe-669f868f7451@{shape: rect, label: 'A3<br>0.0'}
94895ff8-d54c-4b66-9119-a541be208d21@{shape: rect, label: 'A1_2<br>0.0'}
e7c16258-5465-4a41-a7a7-68cba0a632f3-->4a8cdef8-984e-49eb-9e55-ecb7fc0f6ebf
e7c16258-5465-4a41-a7a7-68cba0a632f3-->6c658872-dc91-4e9a-b5fe-669f868f7451
93e3fcf1-3a1e-490e-8d29-57423e576837-->94895ff8-d54c-4b66-9119-a541be208d21
end
subgraph d78d4ec5-c07c-4caa-9073-b769e3b00413[time]
direction LR
2b296eab-eaf5-4070-9c20-37b1820f5ed1@{shape: rect, label: 'Waiting Task<br>0.0'}
969441c7-68fc-43db-8b5b-096d96dcfcca@{shape: rect, label: 'A1 Inst End<br>0.0'}
bf1b7126-9831-4984-85dd-9d7ab4b7eaf4@{shape: rect, label: 'A2 Inst End<br>0.0'}
34cdd9d7-9bd9-4eaf-9afa-3af5d6ef4bcc@{shape: rect, label: 'A1 Inst Start<br>0.0'}
4faed327-63c4-4b3b-a6ff-0b7b0bf44568@{shape: rect, label: 'A2 Inst Start<br>0.0'}
34cdd9d7-9bd9-4eaf-9afa-3af5d6ef4bcc-->969441c7-68fc-43db-8b5b-096d96dcfcca
4faed327-63c4-4b3b-a6ff-0b7b0bf44568-->bf1b7126-9831-4984-85dd-9d7ab4b7eaf4
2b296eab-eaf5-4070-9c20-37b1820f5ed1-->34cdd9d7-9bd9-4eaf-9afa-3af5d6ef4bcc
969441c7-68fc-43db-8b5b-096d96dcfcca-->4faed327-63c4-4b3b-a6ff-0b7b0bf44568
end
subgraph 230ad749-1240-4b9c-aa57-5fc37bd5a470[team1]
direction LR
5cc9c92b-efb3-4cee-b6e6-f6eef2cdb808@{shape: stadium, label: 'workerB'}
44c229bc-52c6-48b7-b12e-d39e9d1f355e@{shape: stadium, label: 'workerA'}
end
subgraph 6cf46da7-37bf-4840-aefb-ede2f5686baa[placeA]
direction LR
2866d497-27a0-48ed-95ab-487f025b82c4@{shape: stadium, label: 'facilityA'}
d0f170ed-4238-4f16-aa4c-89082e2e6361@{shape: stadium, label: 'facilityA_2'}
end
dc27aec0-93e7-485e-8297-476ab7288297-.-4a8cdef8-984e-49eb-9e55-ecb7fc0f6ebf
dc27aec0-93e7-485e-8297-476ab7288297-.-e7c16258-5465-4a41-a7a7-68cba0a632f3
dc27aec0-93e7-485e-8297-476ab7288297-.-93e3fcf1-3a1e-490e-8d29-57423e576837
dc27aec0-93e7-485e-8297-476ab7288297-.-6c658872-dc91-4e9a-b5fe-669f868f7451
dc27aec0-93e7-485e-8297-476ab7288297-.-94895ff8-d54c-4b66-9119-a541be208d21
4a8cdef8-984e-49eb-9e55-ecb7fc0f6ebf-.-230ad749-1240-4b9c-aa57-5fc37bd5a470
6cf46da7-37bf-4840-aefb-ede2f5686baa-.-4a8cdef8-984e-49eb-9e55-ecb7fc0f6ebf
e7c16258-5465-4a41-a7a7-68cba0a632f3-.-230ad749-1240-4b9c-aa57-5fc37bd5a470
6cf46da7-37bf-4840-aefb-ede2f5686baa-.-e7c16258-5465-4a41-a7a7-68cba0a632f3
93e3fcf1-3a1e-490e-8d29-57423e576837-.-230ad749-1240-4b9c-aa57-5fc37bd5a470
6cf46da7-37bf-4840-aefb-ede2f5686baa-.-93e3fcf1-3a1e-490e-8d29-57423e576837
6c658872-dc91-4e9a-b5fe-669f868f7451-.-230ad749-1240-4b9c-aa57-5fc37bd5a470
6cf46da7-37bf-4840-aefb-ede2f5686baa-.-6c658872-dc91-4e9a-b5fe-669f868f7451
94895ff8-d54c-4b66-9119-a541be208d21-.-230ad749-1240-4b9c-aa57-5fc37bd5a470
6cf46da7-37bf-4840-aefb-ede2f5686baa-.-94895ff8-d54c-4b66-9119-a541be208d21
4faed327-63c4-4b3b-a6ff-0b7b0bf44568-->e7c16258-5465-4a41-a7a7-68cba0a632f3
34cdd9d7-9bd9-4eaf-9afa-3af5d6ef4bcc-->93e3fcf1-3a1e-490e-8d29-57423e576837
94895ff8-d54c-4b66-9119-a541be208d21-->969441c7-68fc-43db-8b5b-096d96dcfcca
93e3fcf1-3a1e-490e-8d29-57423e576837-->969441c7-68fc-43db-8b5b-096d96dcfcca
4a8cdef8-984e-49eb-9e55-ecb7fc0f6ebf-->bf1b7126-9831-4984-85dd-9d7ab4b7eaf4
e7c16258-5465-4a41-a7a7-68cba0a632f3-->bf1b7126-9831-4984-85dd-9d7ab4b7eaf4
```

In [ ]:
project.write_simple_json("sample_task_dependency.json")